In [7]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

# Google Colab에서 CSV 파일 불러오기
df = pd.read_csv('/content/Golf_test.csv')

# ------------------------------------------------------------
# 데이터 전처리: after 점수를 긴 형식으로 변환 (ANOVA를 위한 구조)
# ------------------------------------------------------------
df_melted = pd.DataFrame({
    'Score': pd.concat([df['TypeA_after'], df['TypeB_after'], df['TypeC_after']], ignore_index=True),
    'Group': ['TypeA'] * len(df) + ['TypeB'] * len(df) + ['TypeC'] * len(df)
})

# ------------------------------------------------------------
# 1. OLS (Ordinary Least Squares): 회귀 모델을 기반으로 ANOVA 준비
# ------------------------------------------------------------
model = smf.ols('Score ~ C(Group)', data=df_melted).fit()

# ------------------------------------------------------------
# 2. ANOVA table: statsmodels의 anova_lm 사용
# ------------------------------------------------------------
anova_result = anova_lm(model)
print("=== ANOVA table ===")
print(anova_result)

# ------------------------------------------------------------
# 3. f_oneway: SciPy의 단순 One-way ANOVA 함수
# ------------------------------------------------------------
f_stat, p_val = f_oneway(df['TypeA_after'], df['TypeB_after'], df['TypeC_after'])
print("\n=== SciPy f_oneway ===")
print(f"F-statistic: {f_stat:.4f}, p-value: {p_val:.4f}")

# ------------------------------------------------------------
# 4. Tukey HSD (사후 검정): 어떤 그룹 간 차이가 있는지 확인
# ------------------------------------------------------------
tukey = pairwise_tukeyhsd(endog=df_melted['Score'], groups=df_melted['Group'], alpha=0.05)
print("\n=== Tukey HSD 결과 ===")
print(tukey.summary())

=== ANOVA table ===
             df    sum_sq     mean_sq         F    PR(>F)
C(Group)    2.0    910.84  455.420000  5.857876  0.003567
Residual  147.0  11428.50   77.744898       NaN       NaN

=== SciPy f_oneway ===
F-statistic: 5.8579, p-value: 0.0036

=== Tukey HSD 결과 ===
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
 TypeA  TypeB     5.38 0.0076  1.2047  9.5553   True
 TypeA  TypeC     0.32  0.982 -3.8553  4.4953  False
 TypeB  TypeC    -5.06  0.013 -9.2353 -0.8847   True
----------------------------------------------------
